<a href="https://colab.research.google.com/github/sooyen12/pyproject/blob/main/%EC%84%9C%EC%B4%88%EA%B5%AC_CCTV%26%EC%9E%90%EC%A0%84%EA%B1%B0_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import folium
# 엑셀 파일 경로 지정
file_path = '/content/drive/MyDrive/ColabNotebooks/cctv_data.xlsx'

# 엑셀 파일 읽기
df_cctv= pd.read_excel(file_path)

# 첫 번째 몇 줄 확인
df_cctv.head()

/usr/local/lib/python3.11/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,번호,관리기관명,소재지도로명주소,소재지지번주소,설치목적구분,카메라대수,카메라화소수,촬영방면정보,보관일수,설치연월,관리기관전화번호,WGS84위도,WGS84경도,데이터기준일자
0,1,서울시 서초구청,서울특별시 서초구 반포대로121,서울특별시 서초구 서초3동 1541-1,교통단속,1,200,360도 전방면,30,2013-12,02-2155-6087,37.490020,127.007739,2024-06-30
1,2,서울시 서초구청,서울특별시 서초구 사평대로58길,서울특별시 서초구 서초4동 1303-16,교통단속,1,200,360도 전방면,30,2013-03,02-2155-6087,37.502977,127.024068,2024-06-30
2,3,서울시 서초구청,서울특별시 서초구 사평대로58길,서울특별시 서초구 서초4동 1303-16,교통단속,1,200,고정,30,2017-09,02-2155-6087,37.502958,127.024072,2024-06-30
3,4,서울시 서초구청,서울특별시 서초구 사평대로58길,서울특별시 서초구 서초4동 1303-16,교통단속,1,200,고정,30,2017-07,02-2155-6087,37.503000,127.024060,2024-06-30
4,5,서울시 서초구청,서울특별시 서초구 서초대로355,서울특별시 서초구 서초4동 1688-6,교통단속,1,200,360도 전방면,30,2013-12,02-2155-6087,37.496351,127.020210,2024-06-30


In [ ]:
import pandas as pd
import folium
# CSV 파일 경로 (코랩에 직접 업로드한 경우 경로를 맞춰줘야 함)
file_path = '/content/drive/MyDrive/ColabNotebooks/12_23_bicycle.csv'
# CSV 파일 불러오기
df_bicycle = pd.read_csv(file_path, encoding='cp949')

In [ ]:
# 위도, 경도 컬럼 이름
latitude_column = 'WGS84위도'
longitude_column = 'WGS84경도'

# 방면 컬럼 정리 (공백 제거, 문자열 처리)
df_cctv['촬영방면정보'] = df_cctv['촬영방면정보'].astype(str).str.strip()

# 지도 중심을 첫 번째 좌표로 설정
center_lat = df_cctv[latitude_column].iloc[0]
center_lon = df_cctv[longitude_column].iloc[0]

In [ ]:
# '시도시군구명'에 '서울특별시 서초구' 포함된 행만 필터링
df_bicycle_seocho = df_bicycle[df_bicycle['시도시군구명'].str.contains('서초구', na=False)]

# 위도, 경도를 숫자로 변환 (혹시 몰라서)
df_bicycle_seocho['위도'] = pd.to_numeric(df_bicycle_seocho['위도'], errors='coerce')
df_bicycle_seocho['경도'] = pd.to_numeric(df_bicycle_seocho['경도'], errors='coerce')

<ipython-input-41-c3270201c2a1>:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_bicycle_seocho['위도'] = pd.to_numeric(df_bicycle_seocho['위도'], errors='coerce')
<ipython-input-41-c3270201c2a1>:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_bicycle_seocho['경도'] = pd.to_numeric(df_bicycle_seocho['경도'], errors='coerce')


In [ ]:
# 공백 제거 후 문자열에 '서울특별시 송파구'가 포함된 행만 필터링
df_bicycle['시도시군구명'] = df_bicycle['시도시군구명'].astype(str).str.strip()
songpa_df = df_bicycle[df_bicycle['시도시군구명'].str.contains('서울특별시 서초구', na=False)]

In [ ]:
# 자전거 사고 다발지역 좌표 컬럼명
latitude = '위도'
longitude = '경도'

In [ ]:
# 지도 생성
m = folium.Map(location=[37.4836, 127.0326], zoom_start=13)

import folium

# 서초구 중심 좌표
map_center = [37.4836, 127.0326]
m = folium.Map(location=map_center, zoom_start=13)

# 자전거 사고 다발 지역 표시
for _, row in df_bicycle_seocho.iterrows():
    lat = row['위도']
    lon = row['경도']
    accidents = row.get('사고건수', 1)

    # 사고건수 없으면 기본값 1, 크기 키워서 잘 보이게
    folium.Circle(
        location=[lat, lon],
        radius=accidents * 20,
        color='blue',
        fill=True,
        fill_color='blue',
        fill_opacity=0.5,
        popup=f'사고 {accidents}건'
    ).add_to(m)

    # 원 그리기
for _, row in df_cctv.iterrows():
    lat = row[latitude_column]
    lon = row[longitude_column]
    direction = row['촬영방면정보']

    # 360도 전방 여부에 따라 색상 지정
    if '360' in direction:
        color = 'red'
    else:
        color = 'gray'

    folium.Circle(
        location=[lat, lon],
        radius=10,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.3,
        popup=direction
    ).add_to(m)

m

